# FastWoe Piecewise WOE

Author: https://www.github.com/xRiskLab

This notebook demonstrates the **piecewise WOE** approach described by Raymond Anderson (2015) in _Piecewise Logistic Regression: an Application in Credit Scoring_.

Instead of one WOE variable per characteristic, bins are grouped into **pieces** -- each piece becomes its own regression variable with its own coefficient.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from fastwoe import FastWoe

## 1. Load Data and Fit WOE


In [3]:
# Set the path to the data directory
data_dir = Path.cwd().parent.parent / "data"

# Load the data
df = pd.read_csv(data_dir / "BankCaseStudyData.csv")

# Define the features and label
features = [
    "Application_Score",
    "Bureau_Score",
    "Number_of_Payments",
]

label = "Final_Decision"

X = df[features]
y = df[label].map({"Accept": 0, "Decline": 1})

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# Fit FastWoe
woe = FastWoe(
    binning_method="tree",
    tree_kwargs={"max_depth": 3, "min_samples_leaf": 50},
    random_state=42,
)
woe.fit(X_train, y_train)

# Inspect the WOE mappings
for feat in features:
    print(f"\n{feat}:")
    print(woe.get_mapping(feat)[["count", "event_rate", "woe"]])


Application_Score:
   count  event_rate       woe
0   1081    0.740055  3.194555
1    797    0.631117  2.685308
2    145    0.503448  2.162091
3    169    0.384615  1.678294
4    474    0.253165  1.066493
5   2429    0.113627  0.094081
6   2056    0.046693 -0.868053
7  12736    0.011385 -2.315706

Bureau_Score:
   count  event_rate       woe
0    190    0.836842  3.783215
1   1676    0.693914  2.966780
2    129    0.480620  2.070740
3    159    0.377358  1.647523
4    411    0.311436  1.354881
5    245    0.163265  0.514168
6   4868    0.066968 -0.485927
7  12209    0.011467 -2.308455

Number_of_Payments:
   count  event_rate       woe
0   1098    0.643898  2.740621
1   2427    0.128554  0.234491
2   2351    0.093577 -0.122421
3   3458    0.071718 -0.412299
4  10553    0.056003 -0.676419


## 2. Assign Pieces (Sign Strategy)

The default `"sign"` strategy splits each feature into two pieces:

- **Piece 0**: bins with WOE < 0 (higher risk than average)
- **Piece 1**: bins with WOE >= 0 (lower risk than average)


In [5]:
woe.assign_pieces(strategy="sign")

# Show WOE values with their piece assignments
for feat in features:
    print(f"\n{feat}:")
    print(woe.get_mapping(feat)[["woe", "piece"]])


Application_Score:
        woe  piece
0  3.194555      1
1  2.685308      1
2  2.162091      1
3  1.678294      1
4  1.066493      1
5  0.094081      1
6 -0.868053      0
7 -2.315706      0

Bureau_Score:
        woe  piece
0  3.783215      1
1  2.966780      1
2  2.070740      1
3  1.647523      1
4  1.354881      1
5  0.514168      1
6 -0.485927      0
7 -2.308455      0

Number_of_Payments:
        woe  piece
0  2.740621      1
1  0.234491      1
2 -0.122421      0
3 -0.412299      0
4 -0.676419      0


## 3. Compare Standard vs Piecewise Output


In [6]:
# Standard WOE: one column per feature
X_train_woe = woe.transform(X_train, output="woe")
print(f"Standard shape: {X_train_woe.shape}")
print(X_train_woe.head())

print()

# Piecewise WOE: one column per (feature, piece) pair
X_train_pw = woe.transform(X_train, output="piecewise")
print(f"Piecewise shape: {X_train_pw.shape}")
print(X_train_pw.head())

Standard shape: (19887, 3)
       Application_Score  Bureau_Score  Number_of_Payments
20887          -0.868053     -0.485927           -0.412299
9927           -0.868053     -0.485927           -0.412299
6401            1.066493      2.070740            0.234491
24693          -0.868053     -0.485927            0.234491
21153          -0.868053     -0.485927           -0.412299

Piecewise shape: (19887, 6)
       Application_Score__piece_0  Application_Score__piece_1  \
20887                   -0.868053                    0.000000   
9927                    -0.868053                    0.000000   
6401                     0.000000                    1.066493   
24693                   -0.868053                    0.000000   
21153                   -0.868053                    0.000000   

       Bureau_Score__piece_0  Bureau_Score__piece_1  \
20887              -0.485927                0.00000   
9927               -0.485927                0.00000   
6401                0.000000      

## 4. Logistic Regression: Base Case vs Piecewise

Anderson (2015) compared three approaches and found that piecewise consistently achieved higher Gini coefficients, especially on out-of-time samples.


In [8]:
# Transform test set
X_test_woe = woe.transform(X_test, output="woe")
X_test_pw = woe.transform(X_test, output="piecewise")

# --- Base Case: standard WOE ---
lr_base = LogisticRegression(penalty=None, random_state=42)
lr_base.fit(X_train_woe, y_train)

auc_base = roc_auc_score(y_test, lr_base.predict_proba(X_test_woe)[:, 1])
gini_base = 2 * auc_base - 1

print("Base Case (standard WOE)")
print(f"Gini: {gini_base:.4f}")
print("Coefficients:")
for name, coef in zip(X_train_woe.columns, lr_base.coef_[0]):
    print(f"  {name}: {coef:.4f}")

print()

# Piecewise
lr_pw = LogisticRegression(penalty=None, random_state=42)
lr_pw.fit(X_train_pw, y_train)

auc_pw = roc_auc_score(y_test, lr_pw.predict_proba(X_test_pw)[:, 1])
gini_pw = 2 * auc_pw - 1

print("Piecewise WOE")
print(f"Gini: {gini_pw:.4f}")
print("Coefficients:")
for name, coef in zip(X_train_pw.columns, lr_pw.coef_[0]):
    print(f"  {name}: {coef:.4f}")

print(
    f"\nGini lift (piecewise vs base): {(gini_pw - gini_base):.4f} "
    f"({(gini_pw - gini_base) / gini_base * 100:+.2f}%)"
)

Base Case (standard WOE)
Gini: 0.8899
Coefficients:
  Application_Score: 0.8900
  Bureau_Score: 0.4042
  Number_of_Payments: 0.5882

Piecewise WOE
Gini: 0.8884
Coefficients:
  Application_Score__piece_0: 1.4881
  Application_Score__piece_1: 0.6968
  Bureau_Score__piece_0: 0.8172
  Bureau_Score__piece_1: 0.4547
  Number_of_Payments__piece_0: -0.3173
  Number_of_Payments__piece_1: 0.6798

Gini lift (piecewise vs base): -0.0015 (-0.17%)


## 5. Custom Piece Map

You can define your own piece groupings per feature. Features not in the map fall back to the `strategy` parameter.


In [ ]:
# See what bins exist for Application_Score
print("Application_Score bins and WOE values:")
app_mapping = woe.get_mapping("Application_Score")
print(app_mapping[["woe"]])

In [ ]:
# Custom piece map: group Application_Score into 3 risk tiers
# Bins are indexed 0..N-1 matching get_mapping() row order
# Sort by WOE to create meaningful groupings
app_sorted = app_mapping["woe"].sort_values()
n = len(app_sorted)

custom_pieces = {}
for i, (idx, _woe_val) in enumerate(app_sorted.items()):
    if i < n // 3:
        custom_pieces[idx] = 0  # high risk
    elif i < 2 * n // 3:
        custom_pieces[idx] = 1  # mid risk
    else:
        custom_pieces[idx] = 2  # low risk

print("Custom piece assignment for Application_Score:")
for idx, piece in sorted(custom_pieces.items()):
    print(f"  bin {idx} (WOE={app_mapping.loc[idx, 'woe']:.3f}) -> piece {piece}")

# Apply custom map for Application_Score, sign strategy for the rest
woe.assign_pieces(strategy="sign", piece_map={"Application_Score": custom_pieces})

X_train_custom = woe.transform(X_train, output="piecewise")
print(f"\nColumns: {X_train_custom.columns.tolist()}")
print(X_train_custom.head())

## 6. Model Complexity Comparison

Anderson (2015) notes that the piecewise approach increases the number of variables but keeps the number of characteristics the same, striking a balance between model complexity and predictive power.


In [11]:
# Reset to sign strategy for fair comparison
woe.assign_pieces(strategy="sign")

# Count total bins (dummy approach = one variable per bin)
total_bins = sum(len(woe.get_mapping(f)) for f in features)

comparison = pd.DataFrame(
    {
        "Approach": ["Base Case", "Piecewise (sign)", "Dummy"],
        "Characteristics": [len(features)] * 3,
        "Variables": [
            len(features),
            woe.transform(X_train, output="piecewise").shape[1],
            total_bins,
        ],
    }
)
print(comparison.to_string(index=False))

        Approach  Characteristics  Variables
       Base Case                3          3
Piecewise (sign)                3          6
           Dummy                3         21


## Reference

Anderson, R. (2015). _Piecewise Logistic Regression: an Application in Credit Scoring_. Presented at Credit Scoring and Control Conference XIV, Edinburgh, 26-28 August 2015.
